# 05 · Previsões com o modelo vigente e comparação de ciclos

Carrega o modelo vigente pelo manifesto (o `Pipeline` já traz o pré-processamento e o conjunto de features) e
prevê **2025**, o teste reservado: registros que **nunca** participaram da seleção de features, do treino ou da
promoção. Depois compara o ciclo atual com os anteriores e demonstra o alerta do monitor de drift.

In [ ]:
import sys, json, pathlib, warnings
RAIZ = pathlib.Path.cwd().resolve()
while not (RAIZ / "config" / "config.yaml").exists():  # funciona rodando de notebooks/ ou da raiz
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore", message=".*Hint: Inferred schema.*")

import pandas as pd
from src.comum import carregar_config, caminho, estado, ler_ciclo, atualizar_ciclo, configurar_mlflow, tags_ciclo, pasta_evidencias, gatilho
cfg = carregar_config()
pd.set_option("display.width", 160)

In [ ]:
from src import ingestao, modelagem as md, promocao as pr, preprocessamento as pp, drift
ciclo = ler_ciclo(cfg)
vigente, manifesto = pr.carregar_vigente(cfg)
v = manifesto["vigente"]
print(f"vigente: {v['algoritmo']} | ciclo {v['cycle_id']} | {v['data_version']} | features: {v['features']}")

## Previsões para 2025 (teste reservado)

O histórico do trecho de cada dia de 2025 precisa dos dias anteriores, então a base é montada com todos os
acidentes até a véspera (base inicial + todos os lotes + 2025). Só as linhas de 2025 são avaliadas.

In [ ]:
trechos = pd.read_parquet(caminho(cfg["dados"]["trechos"]))
clima = pd.read_parquet(caminho(cfg["dados"]["clima"]))
todos = ingestao.acidentes_limpos(cfg, ingestao.lotes_disponiveis(cfg), incluir_teste=True)
fim = todos["data"].max()
base = pp.montar(pp.tratar(todos, trechos, fim), trechos, clima)
teste = base[base["data"] >= pd.Timestamp(cfg["dados"]["inicio_teste"])].copy()
teste["prob"] = md.prever(vigente, teste, v["features"])
m_teste = md.metricas(teste[pp.ALVO].to_numpy(), teste["prob"].to_numpy())
print(f"{len(teste):,} linhas de 2025 ({teste[pp.ALVO].mean():.2%} com acidente)")
m_teste

In [ ]:
nomes = trechos.set_index("trecho_id")
top = (teste.groupby(["trecho_id", "periodo"]).agg(prob_media=("prob", "mean"), periodos_com_acidente=(pp.ALVO, "sum"))
       .sort_values("prob_media", ascending=False).head(10).reset_index())
top["local"] = top["trecho_id"].map(lambda t: f"BR-{nomes.at[t, 'br']} km {nomes.at[t, 'trecho_km']} ({nomes.at[t, 'municipio']})")
top[["local", "periodo", "prob_media", "periodos_com_acidente"]]

## Previsão linha a linha

O trecho de maior risco no último dia de 2025, nos quatro períodos: as features brutas entram no `Pipeline` e
sai a probabilidade.

In [ ]:
exemplo = teste[(teste["trecho_id"] == top.at[0, "trecho_id"])].tail(4)[["trecho_id", "data", "periodo", *v["features"]]]
exemplo.assign(prob=md.prever(vigente, exemplo, v["features"]))

## Comparação entre ciclos

In [ ]:
hist = pd.DataFrame(manifesto["historico"])
cols = ["cycle_id", "data_version", "gatilho", "lotes_novos", "vencedor_ciclo", "valor_vencedor", "valor_vigente", "decisao", "vigente_apos"]
hist[[c for c in cols if c in hist]]

In [ ]:
if len(hist) >= 2:
    a, b = hist.iloc[-2], hist.iloc[-1]
    print("features que entraram:", sorted(set(b["features_eleitas"]) - set(a["features_eleitas"])) or "nenhuma")
    print("features que saíram:  ", sorted(set(a["features_eleitas"]) - set(b["features_eleitas"])) or "nenhuma")
    print("PR-AUC de validação dos candidatos:")
    print(pd.DataFrame({"anterior": {k: c["metricas"]["pr_auc"] for k, c in a["candidatos"].items()},
                        "atual": {k: c["metricas"]["pr_auc"] for k, c in b["candidatos"].items()}}))

In [ ]:
mlflow = configurar_mlflow(cfg)
runs = mlflow.search_runs(experiment_names=[cfg["projeto"]["experimento_mlflow"]])
cols = ["tags.cycle_id", "tags.data_version", "tags.etapa", "tags.algoritmo", "tags.gatilho", "tags.decisao", "metrics.val_pr_auc"]
runs[[c for c in cols if c in runs]].head(20)

## Demonstração do monitor de drift (critério 10)

O mesmo monitor do notebook 01, aplicado a **abril de 2020** (início da pandemia, com queda de tráfego) contra
abril dos anos anteriores, e a um mês normal de 2024. O primeiro deve disparar o alerta e o segundo não.

In [ ]:
demo = {}
for mes in ["2020-04-01", "2024-04-01"]:
    ref, atual = drift.recorte_mensal(base, pd.Timestamp(mes))
    r = drift.monitorar(ref, atual, pp.CANDIDATAS, cfg)
    demo[mes[:7]] = {"alerta": r["alerta"], "psi_max": r["psi_max"], "razao_taxa": r["razao_taxa"], "motivos": "; ".join(r["motivos"])}
pd.DataFrame(demo).T

In [ ]:
ev = pasta_evidencias(cfg, ciclo)
teste[["trecho_id", "data", "periodo", pp.ALVO, "prob"]].sort_values("prob", ascending=False).head(200).to_csv(ev / "previsoes_2025_top200.csv", index=False)
(ev / "metricas_teste_2025.json").write_text(json.dumps(m_teste, indent=2))
with mlflow.start_run(run_name=f"{ciclo['cycle_id']} · previsões 2025", tags=tags_ciclo(ciclo, etapa="previsao", modelo_vigente=v["cycle_id"])):
    mlflow.log_params({"vigente_cycle_id": v["cycle_id"], "vigente_algoritmo": v["algoritmo"], "vigente_run_id": v["run_id"]})
    mlflow.log_metrics({f"teste_{k}": val for k, val in m_teste.items()})
    mlflow.log_artifact(ev / "previsoes_2025_top200.csv")